<a href="https://colab.research.google.com/github/karsarobert/LLM2026/blob/main/ChatGpt2025_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Chat GPT és más nagy nyelvi modellek alkalmazása
##PTE Gépi tanulás
###9. gyakorlat: RAG alapok – Beágyazások és szemantikus keresés
2026. március 30.

Ez a notebook a **RAG (Retrieval-Augmented Generation)** alapjait mutatja be, különös tekintettel a **beágyazásokra (embeddings)** és a **szemantikus keresésre**.

Az LLM-ek tudása korlátozott: nem ismerik a legfrissebb adatokat, nem tudják elérni a saját dokumentumainkat vagy adatbázisunkat. A RAG ezt a problémát oldja meg: **lekérdezi a releváns információt**, majd azt átadja az LLM-nek kontextusként.

**Technológia:** Hugging Face Sentence Transformers + OpenAi API

**A gyakorlat témái:**
1. Mi a RAG és miért van rá szükség?
2. Beágyazások (embeddings) alapjai
3. Sentence Transformers – Hugging Face
4. Első beágyazás létrehozása
5. Hasonlóság mérése – Cosine Similarity
6. Szemantikus keresés egyszerű példán
7. Dokumentum kollekció feldolgozása

---
## 1. Mi a RAG és miért van rá szükség?

### Az LLM-ek korlátai

A nagy nyelvi modellek (LLM-ek) tudása **statikus**: csak azt tudják, amit a tanítási adatokban láttak, és van egy **tudás-határidő** (knowledge cutoff).

**Problémák:**
- Nem ismerik a legfrissebb eseményeket (pl. tegnapi hírek)
- Nem férnek hozzá privát adatokhoz (céges dokumentumok, adatbázisok)
- Nem tudnak valós időben adatot lekérni
- A hallucináció veszélye (kitalált "tények")

### A megoldás: RAG (Retrieval-Augmented Generation)

A RAG **lekérdezéssel bővíti** az LLM-et: először megkeresi a releváns információt, majd azt kontextusként átadja a modellnek.

```
Kérdés: "Milyen projekten dolgozik most a csapat?"
        ↓
1. LEKÉRDEZÉS (Retrieval)
   Dokumentumok keresése → találatok
        ↓
2. KIEGÉSZÍTÉS (Augmentation)
   Kontextus: [dokumentumok] + kérdés
        ↓
3. GENERÁLÁS (Generation)
   LLM válaszol a kontextus alapján
```

### RAG vs. Fine-tuning

| | RAG | Fine-tuning |
|---|---|---|
| **Mire jó?** | Aktuális/privát adat elérése | Stílus, viselkedés tanítása |
| **Költség** | Alacsony | Magas (GPU, idő) |
| **Frissíthetőség** | Azonnali (új dokumentum hozzáadása) | Újra kell tanítani |
| **Használat** | Dokumentum-alapú Q&A, ügyfélszolgálat | Speciális domain nyelv, creative writing |

**Legtöbb esetben a RAG a jobb választás** – gyors, olcsó, könnyen frissíthető.

---
## 2. Beágyazások (embeddings) alapjai

A RAG kulcsa a **beágyazás (embedding)**: a szöveget **számvektorokká** alakítjuk, amelyek megőrzik a szemantikus jelentést.

### Mi az embedding?

Egy **vektor reprezentáció** – a szöveget számok listájává alakítjuk.

```
Szöveg: "A kutya ugat"
         ↓ embedding modell
Vektor: [0.23, -0.51, 0.87, ..., 0.12]  ← pl. 384 szám
```

**Kulcsfontosság:** hasonló jelentésű szövegek **hasonló vektorokat** kapnak!

```
"A kutya ugat"       → [0.23, -0.51, 0.87, ...]
"Az eb csahol"       → [0.25, -0.49, 0.85, ...]  ← KÖZEL
"Az autó gyorsul"    → [-0.67, 0.31, -0.12, ...] ← TÁVOL
```

### Embedding modellek

Az embedding modell egy **neurális háló**, amelyet úgy tanítottak, hogy hasonló jelentésű mondatok közel kerüljenek egymáshoz a vektortérben.

**Népszerű modellek:**
- `all-MiniLM-L6-v2` – gyors, 384 dimenziós, angol
- `all-mpnet-base-v2` – pontosabb, 768 dimenziós, angol
- `multilingual-e5-small` – többnyelvű (magyar is!)
- `SZTAKI-HLT/hubert-base-cc` – magyar specializált

### Miért fontosak?

A hagyományos kulcsszó-alapú keresés csak **szó szerinti egyezést** keres.
Az embedding-alapú keresés **jelentés szerinti** egyezést talál!

| Kérdés | Dokumentum | Kulcsszó keresés | Embedding keresés |
|---|---|---|---|
| "Hogyan főzzek kávét?" | "A kávékészítés lépései..." | ❌ (nincs "főzzek") | ✅ Találat! |
| "Mi az AI?" | "A mesterséges intelligencia..." | ❌ (nincs "AI") | ✅ Találat! |

---
## 3. Sentence Transformers – Hugging Face

A **Sentence Transformers** a Hugging Face által fejlesztett Python könyvtár, amely egyszerűvé teszi az embedding modellek használatát.

**Előnyei:**
- Egyszerű API – 2-3 sor kód
- Előre tanított modellek ezrei
- CPU-n is gyors (kisebb modellekkel)
- Batch feldolgozás támogatás

### 1. lépés – Telepítés

A `sentence-transformers` csomag telepítése. Ez magával hozza a PyTorch-ot és a Hugging Face transformers-t is.

In [ ]:
!pip install sentence-transformers --quiet

print('sentence-transformers telepítve')

sentence-transformers telepítve


### 2. lépés – Importálás és modell betöltése

Az első betöltéskor a modell letöltődik a Hugging Face-ről (~80 MB az all-MiniLM-L6-v2).
Utána gyorsítótárból töltődik.

In [ ]:
from sentence_transformers import SentenceTransformer

# Modell betöltése – első futáskor letölti
modell = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print('Modell betöltve:', modell)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modell betöltve: SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


### 3. lépés – A modell információi

Nézzük meg a modell dimenzióját (hány számból áll egy vektor) és a maximális token hosszt.

In [ ]:
print('Embedding dimenzió:', modell.get_sentence_embedding_dimension())
print('Max szekvencia hossz:', modell.max_seq_length, 'token')

Embedding dimenzió: 384
Max szekvencia hossz: 256 token


---
## 4. Első beágyazás létrehozása

Most készítsük el első beágyazásunkat! Egy egyszerű mondatból létrehozunk egy 384 dimenziós vektort.

### 1. lépés – Egyszerű mondat beágyazása

Az `.encode()` metódus veszi a szöveget és visszaad egy NumPy vektort.

In [ ]:
mondat = "A mesterséges intelligencia forradalmasítja a világot."

# Beágyazás létrehozása
beagyazas = modell.encode(mondat)

print('Típus:', type(beagyazas))
print('Méret:', beagyazas.shape)
print('Első 10 érték:', beagyazas[:10])

Típus: <class 'numpy.ndarray'>
Méret: (384,)
Első 10 érték: [ 0.02547687  0.08405149 -0.02676535 -0.00727807 -0.07977271 -0.01027037
  0.03082035  0.087253   -0.01595448  0.0325489 ]


### 2. lépés – Normalizálás

A beágyazások **normalizálása** (hossz = 1) egyszerűsíti a hasonlóság számítást.
A `normalize_embeddings=True` paraméter automatikusan normalizál.

In [ ]:
import numpy as np

# Normalizált beágyazás
beagyazas_norm = modell.encode(mondat, normalize_embeddings=True)

# Ellenőrzés: a vektor hossza 1.0 kell legyen
hossz = np.linalg.norm(beagyazas_norm)
print(f'Vektor hossza: {hossz:.6f}')  # ~1.0

Vektor hossza: 1.000000


### 3. lépés – Több mondat egyszerre (batch)

Az `.encode()` listát is elfogad – hatékonyabb, mint egyesével feldolgozni.

In [ ]:
mondatok = [
    "A kutya az ember legjobb barátja.",
    "A macska önálló állat.",
    "Az kutya gyorsan tud futni.",
    "A repülőgép a levegőben száll."
]

# Batch beágyazás – minden mondathoz egy vektor
beagyazasok = modell.encode(mondatok, normalize_embeddings=True)

print('Beágyazások mérete:', beagyazasok.shape)  # (4, 384) – 4 mondat, 384 dimenzió

Beágyazások mérete: (4, 384)


---
## 5. Hasonlóság mérése – Cosine Similarity

A **cosine similarity** (koszinusz hasonlóság) méri, mennyire "néznek ugyanabba az irányba" két vektor.

```
Érték: -1.0 ... +1.0

+1.0  → teljesen azonos irány (nagyon hasonló)
 0.0  → merőleges (semleges)
-1.0  → ellentétes irány (ellentétes)
```

Normalizált vektoroknál a cosine similarity = **dot product** (skalárszorzat).

### 1. lépés – Két mondat hasonlósága

Számítsuk ki a hasonlóságot "kutya" és "macska" mondatok között!

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Beágyazások (már létrehoztuk az előző cellában)
# beagyazasok[0] = "A kutya..."
# beagyazasok[1] = "A macska..."

# Hasonlóság számítása
hasonlosag = cosine_similarity([beagyazasok[0]], [beagyazasok[1]])[0][0]

print(f'Hasonlóság (kutya ↔ macska): {hasonlosag:.4f}')

Hasonlóság (kutya ↔ macska): 0.4500


### 2. lépés – Hasonlósági mátrix

Számítsuk ki az összes mondat közötti hasonlóságot – egy 4×4 mátrixot kapunk.

In [ ]:
# Hasonlósági mátrix – minden mondat vs. minden mondat
hasonlosagi_matrix = cosine_similarity(beagyazasok)

print('Hasonlósági mátrix:')
print(hasonlosagi_matrix)
print()
print('Átló (önmagával): mind 1.0')  # minden mondat önmagával 100% hasonló

Hasonlósági mátrix:
[[1.         0.45002344 0.5546993  0.3409101 ]
 [0.45002344 1.0000001  0.40029335 0.32460493]
 [0.5546993  0.40029335 1.         0.30378556]
 [0.3409101  0.32460493 0.30378556 0.99999994]]

Átló (önmagával): mind 1.0


### 3. lépés – Vizualizáció táblázatban

Rendezzük táblázatba az eredményeket – így jobban látható, melyik mondatok hasonlóak.

In [ ]:
import pandas as pd

# Rövidített nevek a jobb olvashatóságért
nevek = ['Kutya', 'Macska', 'Kutya', 'Repülő']

df_hasonlosag = pd.DataFrame(hasonlosagi_matrix, index=nevek, columns=nevek)
print(df_hasonlosag.round(3))

        Kutya  Macska  Kutya  Repülő
Kutya   1.000   0.450  0.555   0.341
Macska  0.450   1.000  0.400   0.325
Kutya   0.555   0.400  1.000   0.304
Repülő  0.341   0.325  0.304   1.000


### 4. lépés – Leginkább és legkevésbé hasonló párok

Keressük meg a leginkább és legkevésbé hasonló mondatpárokat (az átló kivételével).

In [ ]:
# Átló nullázása (önmagukkal ne hasonlítsunk)
# Maximum kereséshez
matrix_max = hasonlosagi_matrix.copy()
np.fill_diagonal(matrix_max, -np.inf)

max_idx = np.unravel_index(matrix_max.argmax(), matrix_max.shape)
max_ertek = matrix_max[max_idx]

print(f'Leginkább hasonló: {nevek[max_idx[0]]} ↔ {nevek[max_idx[1]]} ({max_ertek:.4f})')
print(f'  → "{mondatok[max_idx[0]]}"')
print(f'  → "{mondatok[max_idx[1]]}"')
print()

# Minimum kereséshez
matrix_min = hasonlosagi_matrix.copy()
np.fill_diagonal(matrix_min, np.inf)

min_idx = np.unravel_index(matrix_min.argmin(), matrix_min.shape)
min_ertek = matrix_min[min_idx]

print(f'Legkevésbé hasonló: {nevek[min_idx[0]]} ↔ {nevek[min_idx[1]]} ({min_ertek:.4f})')
print(f'  → "{mondatok[min_idx[0]]}"')
print(f'  → "{mondatok[min_idx[1]]}"')

Leginkább hasonló: Kutya ↔ Kutya (0.5547)
  → "A kutya az ember legjobb barátja."
  → "Az kutya gyorsan tud futni."

Legkevésbé hasonló: Kutya ↔ Repülő (0.3038)
  → "Az kutya gyorsan tud futni."
  → "A repülőgép a levegőben száll."


---
## 6. Szemantikus keresés egyszerű példán

Most összerakjuk az első **szemantikus keresőnket**!

A folyamat:
```
1. Dokumentumok beágyazása (egyszer)
2. Kérdés beágyazása
3. Hasonlóság számítása
4. Top-K leginkább hasonló dokumentum visszaadása
```

### 1. lépés – Mini tudásbázis létrehozása

10 mondat különböző témákról – ez lesz a "dokumentum kollekcióink".

In [ ]:
dokumentumok = [
    "A mesterséges intelligencia a számítástechnika egyik leggyorsabban fejlődő területe.",
    "A gépi tanulás segítségével a számítógépek adatokból tanulnak mintákat.",
    "A neurális hálózatok az emberi agy működését utánozzák.",
    "Python a legnépszerűbb programozási nyelv a data science területén.",
    "A Jupyter Notebook interaktív fejlesztői környezet Python kódhoz.",
    "A Pécs egy szép város Dél-Magyarországon, gazdag kulturális örökséggel.",
    "A PTE az ország egyik legrégebbi egyeteme, 1367-ben alapították.",
    "A magyarországi egyetemek közül a PTE kiemelkedik orvosképzésében.",
    "A kávé a világ egyik legnépszerűbb itala, reggel sokan fogyasztják.",
    "Az espresso egy tömény kávé típus, amelyet olasz módra készítenek."
]

print(f'{len(dokumentumok)} dokumentum betöltve')

10 dokumentum betöltve


### 2. lépés – Dokumentumok beágyazása

Ez a **lassú lépés** – de csak egyszer kell lefuttatni.
A beágyazásokat el lehet menteni és újra fel lehet használni.

In [ ]:
# Batch beágyazás – mind a 10 dokumentum egyszerre
dok_beagyazasok = modell.encode(dokumentumok, normalize_embeddings=True, show_progress_bar=True)

print(f'Beágyazások kész: {dok_beagyazasok.shape}')  # (10, 384)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Beágyazások kész: (10, 384)


### 3. lépés – Keresési függvény

Készítünk egy függvényt, amely:
1. Beágyazza a kérdést
2. Kiszámolja a hasonlóságokat
3. Visszaadja a top-K legjobb találatot

In [ ]:
def kereses(kerdes, top_k=3):
    """
    Szemantikus keresés a dokumentumok között.

    Args:
        kerdes: a felhasználó által megadott keresőkérdés
        top_k: hány legjobb találatot adjunk vissza

    Returns:
        Lista, amelyben minden elem egy tuple:
        (dokumentum indexe, hasonlósági pontszám, dokumentum szövege)
    """

    # A keresőkérdés szövegét számszerű vektorrá (embeddinggé) alakítjuk.
    # A normalize_embeddings=True miatt a vektor normálva lesz,
    # ami a koszinusz-hasonlóság számításánál előnyös.
    kerdes_beagyazas = modell.encode(kerdes, normalize_embeddings=True)

    # Kiszámítjuk a kérdés és az összes dokumentum közötti koszinusz-hasonlóságot.
    #
    # [kerdes_beagyazas] azért kell, mert a cosine_similarity 2D bemenetet vár:
    # - bal oldal: 1 darab kérdésvektor
    # - jobb oldal: több dokumentumvektor
    #
    # Az eredmény egy 1 x N méretű mátrix lesz,
    # ahol N a dokumentumok száma.
    # A [0] az első (és egyetlen) sort veszi ki,
    # így egy sima 1D tömböt kapunk vissza.
    hasonlosagok = cosine_similarity([kerdes_beagyazas], dok_beagyazasok)[0]

    # A hasonlósági értékek indexeit növekvő sorrendbe rendezzük.
    # Ezután [::-1] segítségével megfordítjuk a sorrendet,
    # így a legnagyobb hasonlóság kerül előre.
    # Végül [:top_k] kiválasztja az első top_k darab legjobb találatot.
    top_indexek = np.argsort(hasonlosagok)[::-1][:top_k]

    # Ebbe a listába gyűjtjük majd az eredményeket.
    eredmenyek = []

    # Végigmegyünk a legjobb találatok indexein.
    for idx in top_indexek:
        # Minden találathoz eltároljuk:
        # - a dokumentum indexét
        # - a hozzá tartozó hasonlósági pontszámot
        # - magát a dokumentum szövegét
        eredmenyek.append((idx, hasonlosagok[idx], dokumentumok[idx]))

    # Visszaadjuk az összegyűjtött találatokat.
    return eredmenyek


# Jelzés, hogy a függvény elkészült
print('Keresési függvény kész')

Keresési függvény kész


### 4. lépés – Keresési tesztek

Próbáljuk ki különböző kérdésekkel! Figyeljük meg, hogy a szemantikus keresés **nem kulcsszavakat** keres, hanem **jelentést**.

In [ ]:
# 1. keresés: AI témában
print('=== Keresés: "Mi az a gépi tanulás?" ===')
talalatok = kereses('Mi az a gépi tanulás?', top_k=3)

for i, (idx, pont, szoveg) in enumerate(talalatok, 1):
    print(f'{i}. [{pont:.4f}] {szoveg}')
print()

=== Keresés: "Mi az a gépi tanulás?" ===
1. [0.5319] A gépi tanulás segítségével a számítógépek adatokból tanulnak mintákat.
2. [0.4626] A neurális hálózatok az emberi agy működését utánozzák.
3. [0.4219] Az espresso egy tömény kávé típus, amelyet olasz módra készítenek.



In [ ]:
# 2. keresés: Python témában
print('=== Keresés: "Milyen programozási nyelvet használjak adatelemzéshez?" ===')
talalatok = kereses('Milyen programozási nyelvet használjak adatelemzéshez?', top_k=3)

for i, (idx, pont, szoveg) in enumerate(talalatok, 1):
    print(f'{i}. [{pont:.4f}] {szoveg}')
print()

=== Keresés: "Milyen programozási nyelvet használjak adatelemzéshez?" ===
1. [0.5905] A kávé a világ egyik legnépszerűbb itala, reggel sokan fogyasztják.
2. [0.5598] A mesterséges intelligencia a számítástechnika egyik leggyorsabban fejlődő területe.
3. [0.5327] Az espresso egy tömény kávé típus, amelyet olasz módra készítenek.



In [ ]:
# 3. keresés: Pécs témában
print('=== Keresés: "Hol található a PTE?" ===')
talalatok = kereses('Hol található a PTE?', top_k=3)

for i, (idx, pont, szoveg) in enumerate(talalatok, 1):
    print(f'{i}. [{pont:.4f}] {szoveg}')
print()

=== Keresés: "Hol található a PTE?" ===
1. [0.4532] Az espresso egy tömény kávé típus, amelyet olasz módra készítenek.
2. [0.4328] A PTE az ország egyik legrégebbi egyeteme, 1367-ben alapították.
3. [0.3936] A magyarországi egyetemek közül a PTE kiemelkedik orvosképzésében.



In [ ]:
# 4. keresés: irreleváns kérdés
print('=== Keresés: "Hogyan készítsek pizzát?" ===')
talalatok = kereses('Hogyan készítsek pizzát?', top_k=3)

for i, (idx, pont, szoveg) in enumerate(talalatok, 1):
    print(f'{i}. [{pont:.4f}] {szoveg}')

print('\n→ Figyeljük meg: az irreleváns kérdésnél is vannak találatok,')
print('  de a pontszámok alacsonyak (általában < 0.3 = gyenge kapcsolat)')

=== Keresés: "Hogyan készítsek pizzát?" ===
1. [0.5559] A magyarországi egyetemek közül a PTE kiemelkedik orvosképzésében.
2. [0.5552] A kávé a világ egyik legnépszerűbb itala, reggel sokan fogyasztják.
3. [0.5436] Az espresso egy tömény kávé típus, amelyet olasz módra készítenek.

→ Figyeljük meg: az irreleváns kérdésnél is vannak találatok,
  de a pontszámok alacsonyak (általában < 0.3 = gyenge kapcsolat)


---
## 7. Dokumentum kollekció feldolgozása

Nagyobb dokumentum gyűjteményeknél a beágyazásokat érdemes **elmenteni**, hogy ne kelljen mindig újraszámolni őket.

### 1. lépés – Beágyazások mentése fájlba

NumPy `.npy` formátumban mentjük – gyors és hatékony.

In [ ]:
import numpy as np
import json

# Beágyazások mentése
np.save('dokumentum_beagyazasok.npy', dok_beagyazasok)

# Dokumentumok mentése JSON-ba (a szövegeket is el kell menteni!)
with open('dokumentumok.json', 'w', encoding='utf-8') as f:
    json.dump(dokumentumok, f, ensure_ascii=False, indent=2)

print('Beágyazások és dokumentumok elmentve')
print(f'  Beágyazások mérete: {dok_beagyazasok.nbytes / 1024:.1f} KB')

Beágyazások és dokumentumok elmentve
  Beágyazások mérete: 15.0 KB


### 2. lépés – Visszatöltés

A mentett beágyazások gyorsan betölthetők – nem kell újra futtatni a modellt.

In [ ]:
# Visszatöltés
betoltott_beagyazasok = np.load('dokumentum_beagyazasok.npy')

with open('dokumentumok.json', 'r', encoding='utf-8') as f:
    betoltott_dokumentumok = json.load(f)

print('Visszatöltve:')
print(f'  Beágyazások: {betoltott_beagyazasok.shape}')
print(f'  Dokumentumok: {len(betoltott_dokumentumok)} db')

# Ellenőrzés: ugyanazok?
print(f'\nAzonos: {np.allclose(dok_beagyazasok, betoltott_beagyazasok)}')

Visszatöltve:
  Beágyazások: (10, 384)
  Dokumentumok: 10 db

Azonos: True


### 3. lépés – Nagyobb dokumentum kollekció szimulálása

Készítsünk egy 50 dokumentumos adatbázist szakdolgozat témákról.

In [ ]:
# Témakörök bővítése
nagy_kollekcio = [
    # AI / ML
    "A mesterséges intelligencia a számítástechnika egyik leggyorsabban fejlődő területe.",
    "A gépi tanulás segítségével a számítógépek adatokból tanulnak mintákat.",
    "A neurális hálózatok az emberi agy működését utánozzák.",
    "A deep learning több rétegű neurális hálókat használ komplex minták felismerésére.",
    "A reinforcement learning játékokban és robotikában használatos.",
    "A természetes nyelvfeldolgozás (NLP) szövegek megértésével foglalkozik.",
    "A számítógépes látás képek és videók elemzésére szolgál.",
    "A GPT modellek transformer architektúrán alapulnak.",
    "Az embedding vektorok szemantikus információt kódolnak.",
    "A RAG rendszerek lekérdezéssel bővítik az LLM tudását.",

    # Programozás
    "Python a legnépszerűbb programozási nyelv a data science területén.",
    "A Jupyter Notebook interaktív fejlesztői környezet Python kódhoz.",
    "A pandas könyvtár táblázatos adatok kezelésére szolgál.",
    "A NumPy numerikus számításokhoz nyújt gyors tömbműveleteket.",
    "A scikit-learn klasszikus gépi tanulás algoritmusokat tartalmaz.",
    "A TensorFlow és PyTorch deep learning keretrendszerek.",
    "A Git verziókezelő rendszer a kódok nyomon követésére.",
    "A Docker konténerizációt biztosít alkalmazásokhoz.",
    "A FastAPI modern Python web API keretrendszer.",
    "A SQL adatbázis lekérdező nyelv relációs adatbázisokhoz.",

    # Egyetem / Pécs
    "A Pécs egy szép város Dél-Magyarországon, gazdag kulturális örökséggel.",
    "A PTE az ország egyik legrégebbi egyeteme, 1367-ben alapították.",
    "A magyarországi egyetemek közül a PTE kiemelkedik orvosképzésében.",
    "A Pécsi Tudományegyetem 10 karra tagozódik.",
    "A PTE Természettudományi Karán tanulható informatika.",
    "A pécsi egyetem nemzetközi programokat is kínál.",
    "A Zsolnay negyedben található a PTE BTK épülete.",
    "A PMMIK a Pollack Mihály Műszaki és Informatikai Kar.",
    "A Pécs Európa Kulturális Fővárosa volt 2010-ben.",
    "A pécsi Dzsámi a török hódoltság emléke.",

    # Adattudomány
    "Az adattudomány adatokból nyeri ki a hasznos tudást.",
    "A statisztika az adatelemzés alapja.",
    "A nagy adatok (Big Data) speciális feldolgozási módszereket igényelnek.",
    "Az ETL folyamat: Extract, Transform, Load.",
    "A feature engineering javítja a modellek teljesítményét.",
    "A cross-validation modellértékelési technika.",
    "Az overfitting esetén a modell túltanulja a tréning adatokat.",
    "A confusion matrix osztályozási eredményeket vizualizál.",
    "A ROC görbe a klasszifikátorok teljesítményét mutatja.",
    "A hiperparaméter hangolás javítja a modell pontosságát.",

    # Web / mobil
    "A React JavaScript könyvtár felhasználói felületek építéséhez.",
    "A REST API HTTP protokollon keresztül kommunikál.",
    "A JSON az API-k leggyakoribb adatformátuma.",
    "A Flutter platformfüggetlen mobil alkalmazások fejlesztésére szolgál.",
    "A HTML a weboldalak struktúráját definiálja.",
    "A CSS a weboldalak stílusáért felel.",
    "A JavaScript a weboldalak interaktivitását biztosítja.",
    "A Node.js szerver oldali JavaScript futtatókörnyezet.",
    "A MongoDB NoSQL dokumentum adatbázis.",
    "A GraphQL alternatíva a REST API-khoz.",
]

print(f'{len(nagy_kollekcio)} dokumentum betöltve')

50 dokumentum betöltve


### 4. lépés – Nagy kollekció beágyazása

Ez most már több időt vehet igénybe. A `show_progress_bar=True` mutatja a haladást.

In [ ]:
import time

print('Beágyazás indítása...')
start = time.time()

nagy_beagyazasok = modell.encode(nagy_kollekcio, normalize_embeddings=True, show_progress_bar=True)

elapsed = time.time() - start
print(f'\nKész: {nagy_beagyazasok.shape} – {elapsed:.2f} másodperc ({len(nagy_kollekcio)/elapsed:.1f} dok/sec)')

Beágyazás indítása...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Kész: (50, 384) – 1.08 másodperc (46.2 dok/sec)


### 5. lépés – Keresés a nagy kollekcióban

Frissítsük a keresési függvényt, hogy az új kollekcióban keressen.

In [ ]:
def nagy_kereses(kerdes, top_k=5):
    """Keresés a nagy dokumentum kollekcióban."""
    kerdes_beagyazas = modell.encode(kerdes, normalize_embeddings=True)
    hasonlosagok = cosine_similarity([kerdes_beagyazas], nagy_beagyazasok)[0]
    top_indexek = np.argsort(hasonlosagok)[::-1][:top_k]

    eredmenyek = []
    for idx in top_indexek:
        eredmenyek.append((idx, hasonlosagok[idx], nagy_kollekcio[idx]))
    return eredmenyek

# Teszt
print('=== Keresés: "transformer architektúra" ===')
for i, (idx, pont, szoveg) in enumerate(nagy_kereses('transformer architektúra', top_k=3), 1):
    print(f'{i}. [{pont:.4f}] {szoveg}')

=== Keresés: "transformer architektúra" ===
1. [0.7591] A GPT modellek transformer architektúrán alapulnak.
2. [0.3378] A Git verziókezelő rendszer a kódok nyomon követésére.
3. [0.3019] A Pécs Európa Kulturális Fővárosa volt 2010-ben.


In [ ]:
# töltsük be a  karsar/paraphrase-multilingual-MiniLM-L12-hu-v3 embedding modellt!